# EVASPA processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from evaspa.api import run_evaspa
from evaspa.config import check_config
from evaspa.ef import EFModel  # noqa TC001

In [ ]:
def plot_rgb(data: xr.Dataset, bands: list[str] | None = None):
    if bands is None:
        bands = ["red", "green", "blue"]
    rgb = xr.DataArray(
        np.dstack(
            (data[bands[0]].data, data[bands[1]].data, data[bands[2]].data)
        ),
        data.coords.assign(band=["r", "g", "b"]),
        dict(data.sizes, band=3),
    )
    rgb.plot.imshow(  # type: ignore
        rgb="band",
        vmin=rgb.quantile(0.01),
        vmax=rgb.quantile(0.99),
    )
    plt.title("RGB")


def plot_band(data: xr.Dataset, band: str, percentile: float = 0.0):
    data[band].plot.imshow(  # type: ignore
        vmin=data[band].quantile(percentile),
        vmax=data[band].quantile(1 - percentile),
    )


def plot_mask(data: xr.Dataset, mask: str, invert=False):
    cmap = [(0.0, 0.0, 0.0, 0.0)]
    if len(np.unique(data[mask].data)) > 1:
        cmap = [(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)]
    if invert:
        cmap = cmap[::-1]
    valid_cmap = ListedColormap(cmap)
    data[mask].plot(cmap=valid_cmap, add_colorbar=False)


def plot(
    data: xr.Dataset,
    var_name: str | None = None,
    model: EFModel | None = None,
    save: bool = False,
):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    fig = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    if var_name is None and model is None:
        msg = "Provide either var_name or model"
        raise ValueError(msg)
    lst = data["lst"].data

    if var_name is not None:
        var = data[var_name].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # plot edges
    elif model is not None:
        var_name = model.var
        var = data[model.var].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)

        var_arr = np.linspace(var_min, var_max, num=20)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)

        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    # title
    ax.set_title("Evaporative fraction method", fontsize=12)

    # savefig
    if save:
        destfile = "ef.png"
        fig.savefig(destfile)

    plt.show()

## Case: Simple execution

In [ ]:
config_dict = {
    "input": {
        "path": "../tests/data/modis_test_geo.tif",
        "date": "2018-05-16T10:00:00-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "ef": {
            "models": "default_evaspa",
        },
    },
}

#### Execution

In [ ]:
config = check_config(config_dict)

In [ ]:
pprint(config, sort_dicts=False)

In [ ]:
inst_xr, daily_xr = run_evaspa(config["input"], config["params"])

In [ ]:
inst_xr

In [ ]:
daily_xr

In [ ]:
plot_band(inst_xr, band="ef")

In [ ]:
plot_band(daily_xr, band="et")

## Case: Detailed configuration

#### Configuration

In [ ]:
config_dict = {
    "input": {
        "path": "../tests/data/modis_test_geo.tif",
        "date": "2018-05-16T10:00:00-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "filtering": {},
        "ef": {
            "check": {"threshold": 0.02},
            "models": [
                {
                    "name": "model1",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_nb": 20,
                            "percentile": [98, 100],
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_nb": 20,
                            "percentile": [0, 2],
                            "selection": "median",
                        },
                    },
                    "var": "albedo",
                },
                {
                    "name": "model2",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": [95, 100],
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatEdge",
                        "config": {"selection": "min"},
                    },
                    "var": "lai",
                },
            ],
            "options": {"selection": False, "merging": "mean"},
        },
    },
}

In [ ]:
config_dict

In [ ]:
pprint(config_dict, sort_dicts=False)

#### Execution

In [ ]:
config = check_config(config_dict)

In [ ]:
inst_xr, daily_xr = run_evaspa(config["input"], config["params"])

In [ ]:
plot_band(daily_xr, band="et")

## Case: detailed pipeline

In [ ]:
from evaspa import daily, ef, filter, io, seb
from evaspa.config import ParamsConfig

Configuration

In [ ]:
config_dict = {
    "input": {
        "path": "../tests/data/modis_test_geo.tif",
        "date": "2018-05-16T10:00:00-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "filtering": {},
        "ef": {
            "check": {"threshold": 0.02},
            "models": [
                {
                    "name": "model1",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_nb": 20,
                            "percentile": [98, 100],
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_nb": 20,
                            "percentile": [0, 2],
                            "selection": "median",
                        },
                    },
                    "var": "albedo",
                },
                {
                    "name": "model2",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": [95, 100],
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatEdge",
                        "config": {"selection": "min"},
                    },
                    "var": "lai",
                },
            ],
            "options": {"selection": False, "merging": "mean"},
        },
    },
}

Check configuration

In [ ]:
config = check_config(config_dict)
# Validate parameters config
params_config = ParamsConfig.model_validate(config["params"])

Read input data

In [ ]:
data = io.read_input(config["input"])

In [ ]:
data

In [ ]:
plot_band(data, band="lst")

Filter data

In [ ]:
data["valid"] = filter.determine_valid_pixels(
    data, **params_config.filtering.model_dump()
)

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    plot_mask(data=data, mask="valid", invert=True)

Check variability

In [ ]:
ef.check_variability(
    data["lst"], mask=data["valid"], **params_config.ef.check.model_dump()
)

Compute EF

In [ ]:
models, options = ef.initialize(params_config.ef.model_dump())
ef_xr, inst_xr = ef.run(models, data, mask="valid", **options)

In [ ]:
plot_band(inst_xr, band="ef")

Compute LE

In [ ]:
le_xr, merged_xr = seb.run(data, ef_xr, **params_config.seb.model_dump())
inst_xr = merged_xr.merge(
    inst_xr, join="override", combine_attrs="no_conflicts"
)

In [ ]:
le_xr

In [ ]:
plot_band(le_xr, band="model1_rn_kustas")

In [ ]:
plot_band(inst_xr, band="le")

In [ ]:
plot_band(inst_xr, band="et")

Extrapolate at daily scale

In [ ]:
daily_xr = daily.extrapolate_at_daily_scale(
    inst_xr, variables=["le", "et"], **params_config.daily.model_dump()
)

In [ ]:
plot_band(daily_xr, band="le")

In [ ]:
plot_band(daily_xr, band="et")